```{contents}
```
## Validation and Metrics


Training loss alone does **not** indicate whether a model is learning generalizable patterns.
Deep learning systems require **validation protocols** and **evaluation metrics** to:

* Estimate generalization performance
* Detect overfitting and underfitting
* Compare competing models
* Guide hyperparameter tuning and early stopping
* Support deployment decisions

---

### Core Concepts

**Training Set**
Used to optimize model parameters.

**Validation Set**
Used to tune hyperparameters and monitor generalization during training.

**Test Set**
Used once at the end for unbiased performance reporting.

**Overfitting**
Low training error, high validation error.

**Underfitting**
High error on both training and validation.

---

### Validation Strategies

#### Hold-Out Validation

| Split      | Purpose              |
| ---------- | -------------------- |
| Train      | Learn parameters     |
| Validation | Tune hyperparameters |
| Test       | Final evaluation     |

Typical ratios: **70/15/15** or **80/10/10**

#### K-Fold Cross-Validation (mainly for small datasets)

* Data split into *K* folds
* Each fold used once as validation
* Performance averaged across folds

---

### Metrics: What and Why

Metrics quantify **how good** the model predictions are.

#### Classification Metrics

| Metric    | Formula              | Interpretation                      |
| --------- | -------------------- | ----------------------------------- |
| Accuracy  | (TP+TN)/(Total)      | Overall correctness                 |
| Precision | TP/(TP+FP)           | Reliability of positive predictions |
| Recall    | TP/(TP+FN)           | Coverage of true positives          |
| F1        | 2PR/(P+R)            | Precision–Recall balance            |
| ROC-AUC   | Area under ROC curve | Ranking quality                     |

#### Regression Metrics

| Metric | Formula      | Interpretation         |   |                    |
| ------ | ------------ | ---------------------- | - | ------------------ |
| MSE    | mean((y−ŷ)²) | Penalizes large errors |   |                    |
| MAE    | mean(        | y−ŷ                    | ) | Robust to outliers |
| R²     | 1 − SSE/SST  | Variance explained     |   |                    |

---

### Training Workflow with Validation

1. Initialize model and optimizer
2. Train on training set
3. After each epoch:

   * Disable gradient computation
   * Evaluate on validation set
   * Compute metrics
4. Monitor curves: training vs validation
5. Apply early stopping / model selection
6. Final evaluation on test set

---

### PyTorch Demonstration (Classification)

#### Dataset and Model



In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

X = torch.randn(1000, 20)
y = (X.sum(dim=1) > 0).long()

train_X, val_X = X[:800], X[800:]
train_y, val_y = y[:800], y[800:]

train_loader = DataLoader(TensorDataset(train_X, train_y), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(val_X, val_y), batch_size=64)

model = nn.Sequential(
    nn.Linear(20, 64),
    nn.ReLU(),
    nn.Linear(64, 2)
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()




#### Training Loop with Validation Metrics



In [2]:
def evaluate(model, loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0

    with torch.no_grad():
        for Xb, yb in loader:
            logits = model(Xb)
            loss = criterion(logits, yb)
            loss_sum += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

    return loss_sum / len(loader), correct / total

for epoch in range(20):
    model.train()
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        optimizer.step()

    val_loss, val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch}: Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")


Epoch 0: Val Loss=0.5850, Val Acc=0.7600
Epoch 1: Val Loss=0.4902, Val Acc=0.8400
Epoch 2: Val Loss=0.3973, Val Acc=0.8850
Epoch 3: Val Loss=0.3168, Val Acc=0.9050
Epoch 4: Val Loss=0.2537, Val Acc=0.9300
Epoch 5: Val Loss=0.2093, Val Acc=0.9500
Epoch 6: Val Loss=0.1791, Val Acc=0.9650
Epoch 7: Val Loss=0.1560, Val Acc=0.9650
Epoch 8: Val Loss=0.1384, Val Acc=0.9600
Epoch 9: Val Loss=0.1259, Val Acc=0.9600
Epoch 10: Val Loss=0.1161, Val Acc=0.9700
Epoch 11: Val Loss=0.1077, Val Acc=0.9650
Epoch 12: Val Loss=0.0990, Val Acc=0.9700
Epoch 13: Val Loss=0.0952, Val Acc=0.9700
Epoch 14: Val Loss=0.0886, Val Acc=0.9700
Epoch 15: Val Loss=0.0840, Val Acc=0.9700
Epoch 16: Val Loss=0.0808, Val Acc=0.9750
Epoch 17: Val Loss=0.0786, Val Acc=0.9750
Epoch 18: Val Loss=0.0749, Val Acc=0.9750
Epoch 19: Val Loss=0.0727, Val Acc=0.9750




---

### Interpreting Validation Curves

| Pattern                  | Diagnosis                                 |
| ------------------------ | ----------------------------------------- |
| Training ↓, Validation ↓ | Healthy learning                          |
| Training ↓, Validation ↑ | Overfitting                               |
| Both high                | Underfitting                              |
| Validation noisy         | Learning rate too high or batch too small |

---

### Advanced Practices

#### Early Stopping

Stop training when validation loss fails to improve for *N* epochs.

#### Metric-Driven Checkpointing

Save model when validation metric improves.

#### Task-Specific Metrics

| Task                      | Key Metrics              |
| ------------------------- | ------------------------ |
| Medical diagnosis         | Recall, ROC-AUC          |
| Search & ranking          | MAP, NDCG                |
| Imbalanced classification | F1, Precision–Recall AUC |
| Regression forecasting    | MAE, RMSE                |

---

### Summary

| Component     | Purpose                             |
| ------------- | ----------------------------------- |
| Validation    | Generalization monitoring           |
| Metrics       | Quantitative performance assessment |
| Workflow      | Guides training decisions           |
| PyTorch tools | Enable reproducible evaluation      |

A deep learning model is not defined by its training loss, but by **how well it generalizes as measured by validation metrics**.
